

# A General Framework for LP Models in Pyomo (blank)
### OPIM 5641 - Business Decision Modeling · Module 3
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/6_Pyomo_LP/0_General_Framework_LP_Pyomo_blank.ipynb)

*Run me top to bottom - **Runtime → Run all**. Pyomo and a solver install in the first cell.*

The purpose of this script is to give you a general format for coding up LP models in Pyomo. If you can repeat the major elements in this script then you will be on your way toward optimization modeling greatness! Of course, we will use chairs, desks and tables here for our example.

This is the notebook that turns you from someone who *solves* LPs by hand into someone who *builds* them in code - which is how it's actually done on the job. Nail this skeleton and you can point a solver at problems way too big to ever grind out by hand.

🔷 **The nugget:** every Pyomo model is the same handful of pieces - a `ConcreteModel`, your `Var`s, one `Objective`, your `Constraint`s - then you hand it to a solver. Learn the skeleton once and every new problem is just filling in the blanks.

<!-- oct8 -->
## The five steps: your checklist for every LP in Pyomo

| Step | What you do | The code |
|---|---|---|
| **1** | Import Pyomo (and download the solver) | the setup cell below - run it every time |
| **2** | Create the model | `model = ConcreteModel()` |
| **3** | Declare the decision variables | `model.chairs = Var(domain=NonNegativeReals)` |
| **4** | Declare the objective | `model.profit = Objective(expr = 15*model.chairs + ..., sense = maximize)` |
| **5** | Declare the constraints | `model.constraint1 = Constraint(expr = 4*model.chairs + ... <= 1850)` |

Then **solve** it and **read the results**:
`SolverFactory('cbc', executable='/usr/bin/cbc').solve(model)`, then `model.profit()`, `model.chairs()` and so on.

Every model in this class follows these five steps, in this order. Only the numbers change.

# Sample Word Problem

# Example: Furniture
*Section 9.2 (Powell) - Chairs, Desks and Tables*

**Problem Description:**
Veerman Furniture Company makes three kinds of office furniture: chairs, desks, and tables. Each product requires some labor in the parts fabrication department, the assembly department, and the shipping department. The furniture is sold through a regional distributor, who has estimated the maximum potential sales for each product in the coming quarter. Finally,the accounting department has provided some data showing the profit contributions on each product.The decision problem isto determine the product mix - that is, to maximize Veerman’s profit for the quarter by choosing production quantities for the chairs,desks,and tables.

The following data summarizes the parameters of the problem:

| Department | Chairs | Desks | Tables | Hours Available |
|---|---|---|---|---|
| Fabrication | 4 | 6 | 2 | 1,850 |
| Assembly | 3 | 5 | 7 | 2,400 |
| Shipping | 3 | 2 | 4 | 1,500 |
| **Demand Potential** | 360 | 300 | 100 | - |
| **Profit (USD)** | 15 | 24 | 18 | - |

**Define the Objective Function**

$Profit = 15C + 24D + 18T$

**Write the Constraints**

$Max(Z) = 15C + 24D + 18T$

subject to:
* $4C + 6D + 2T <= 1,850$
* $3C + 5D + 7T <= 2,400$
* $3C + 2D + 4T <= 1,500$ 
* $C        <= 360$ 
* $D      <=300$ 
* $T <=100$

Great! Now that your problem is defined - go code it up and solve it.

## Import modules (download Pyomo)
You must do this every time! During class, we may use different solvers depending on the type of optimization problem we are trying to solve. 

In [ ]:
# import modules

# this makes beautiful plots using pylab (matplotlib)
%matplotlib inline
from pylab import * # * means import ALL NAME SPACES

# useful modules for downloading pyomo onto Colab
import shutil
import sys
import os.path

# install pyomo if it doesn't exist
if not shutil.which("pyomo"):
    !pip install -q pyomo
    assert(shutil.which("pyomo"))

# install the 'cbc' solve if it doesn't exist
if not (shutil.which("cbc") or os.path.isfile("cbc")):
    if "google.colab" in sys.modules:
        !apt-get install -y -qq coinor-cbc
    else:
        try:
            !conda install -c conda-forge coincbc 
        except:
            pass

# make sure 'cbc' is the solver that we will invoke later on
assert(shutil.which("cbc") or os.path.isfile("cbc"))

# import ALL NAMESPACES (variables) from pyomo
from pyomo.environ import *

### A note about the asterisk...

**Source:** https://stackoverflow.com/questions/2360724/what-exactly-does-import-import

This practice (of importing * into the current namespace) is however discouraged because it

* provides the opportunity for namespace collisions (say if you had a variable name pi prior to the import)
* may be inefficient

The alternative way to use `pyomo` is as follows (from p.3 in 'Pyomo - Optimization Modeling in Python (3rd Edition)'



```
import pyomo.environ as pyo

model = pyo.ConcreteModel()
model.x_1 = pyo.Var(within=pyo.NonNegativeReals)
model.x_2 = pyo.Var(within=pyo.NonNegativeReals)
model.obj = pyo.Objective(expr=model.x_1 + 2*model.x_2)
model.con1 = pyo.Constraint(expr=3*model.x_1 + 4*model.x_2)
model.con2 = pyo.Constraint(expr=2*model.x_1 + 5*model.x_2)
```

And so on... but honestly, I've never really had an issue with overwriting namespaces. But if you want to code like this for your homework, feel free. Not required.



# ConcreteModel()
In our class, we will typically use the `ConcreteModel()` instead of `AbstractModel()` because it's easier to work with

### A note about $\LaTeX$ 
As a best practice, why not write our the formulas in LaTeX so it's easy for a layman to read your logic? :)

# Declare Decision Variables
You get to name the variables and choose the domain of the decision variables. At this point, you could force an integer solution if you wanted to...

* bounds = A function (or Python object) that gives a (lower,upper) bound pair for the variable
* domain = A set that is a super-set of the values the variable can take on.
* initialize = A function (or Python object) that gives a starting value for the variable; this is particularly important for non-linear models


See here for more: https://pyomo.readthedocs.io/en/stable/pyomo_modeling_components/Variables.html

#  Objective Function
You can choose to `maximize` or `minimize` the objective function - remember, you can't do both! Given these are linear programs, no exponents, absolute value or square roots etc. allowed!

See more here: https://pyomo.readthedocs.io/en/stable/pyomo_modeling_components/Objectives.html

You can name your objective function anything you want - you don't have to call it `model.profit`! You can generalize and call it `model.z` or `model.Obj`. 

# Constraints
You have the freedom to name your constraints ANYTHING you want! This approach works, but you can try renaming `model.Constraint1` to `model.FabricationConstraint` and try to run it, too.

See more here: https://pyomo.readthedocs.io/en/stable/pyomo_modeling_components/Constraints.html

<!-- oct8 -->
### Three kinds of constraints: `<=`, `>=` and `==`

| Math | Pyomo | Example | Use it for |
|---|---|---|---|
| $\le$ | `<=` | `model.constraint1 = Constraint(expr = 4*model.chairs + 6*model.desks + 2*model.tables <= 1850)` | a limit you cannot go over (hours, budget, a demand ceiling) |
| $\ge$ | `>=` | `model.constraint7 = Constraint(expr = model.tables >= 20)` | a minimum you must meet (a contract, a nutrition rule) |
| $=$ | `==` | `model.constraint8 = Constraint(expr = model.chairs == 4*model.tables)` | something that must be **exactly** true (sold in sets: 4 chairs with every table) |

Three things to remember:

- **Equality is a double equals sign, `==`.** A single `=` means "store this value" in Python, so
  `expr = model.chairs = 4*model.tables` is an error.
- Pyomo only takes `<=`, `>=` and `==`. There is **no** strict `<` or `>` in an LP.
- You can **mix** all three kinds in one model. Want to see `==` in action? Add the `constraint8` line
  above to your constraints cell, then re-run the model and solve cells. What happens to the profit?

# [optional] Pretty Print (pprint)
Optional, but you should probably run this to check your work! Especially if you get an error.

3 Var Declarations
    c : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :  None :  None : False :  True : NonNegativeReals
    d : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :  None :  None : False :  True : NonNegativeReals
    t : Size=1, Index=None
        Key  : Lower : Value : Upper : Fixed : Stale : Domain
        None :     0 :  None :  None : False :  True : NonNegativeReals

1 Objective Declarations
    profit : Size=1, Index=None, Active=True
        Key  : Active : Sense    : Expression
        None :   True : maximize : 15*c + 24*d + 18*t

6 Constraint Declarations
    Constraint1 : Size=1, Index=None, Active=True
        Key  : Lower : Body            : Upper  : Active
        None :  -Inf : 4*c + 6*d + 2*t : 1850.0 :   True
    Constraint2 : Size=1, Index=None, Active=True
        Key  : Lower : Body            : Upper  : Active
        None :  -Inf : 3*c + 

# Solve!
Now the moment of truth - two parts you need to pay attention to - the name of the solver (`cbc`) and the path to the solver (`executable='/usr/bin/cbc'`).

**Where is the solver?** The setup cell at the top downloads `cbc` with `apt-get install coinor-cbc`. On Colab,
programs installed that way go in the folder **`/usr/bin/`**, so the solver lives at **`/usr/bin/cbc`**. That is
the path you give `SolverFactory`. Don't believe it? Run the next cell: it asks the computer where `cbc` is.

In [ ]:
# where did the setup cell put the solver? (on Colab this prints /usr/bin/cbc)
print(shutil.which("cbc"))

# ==========================================================
# = Solver Results                                         =
# ==========================================================
# ----------------------------------------------------------
#   Problem Information
# ----------------------------------------------------------
Problem: 
- Name: unknown
  Lower bound: 8400.0
  Upper bound: 8400.0
  Number of objectives: 1
  Number of constraints: 7
  Number of variables: 4
  Number of nonzeros: 3
  Sense: maximize
# ----------------------------------------------------------
#   Solver Information
# ----------------------------------------------------------
Solver: 
- Status: ok
  User time: -1.0
  System time: 0.0
  Wallclock time: 0.0
  Termination condition: optimal
  Termination message: Model was solved to optimality (subject to tolerances), and an optimal solution is available.
  Statistics: 
    Branch and bound: 
      Number of bounded subproblems: None
      Number of created s

After you run the model, it will have a 'termination message' that reads `Termination message: Model was solved to optimality (subject to tolerances), and an optimal solution is available.`

If it says this message, you solved! If it doesn't you probably have a bug in your code (invalid constraints, a typo in your objective function, etc.)

# Inspect the Results

## Objective Function
This the optimal solution

Profit =  8400.0  per week


The use of the `print()` statements makes your output look nice - always do this!!!

## Decision Variables
Note how we need `()` after the variable - lots of students forget this and get a bug!

Chairs =  0.0  units per week
Desks =  275.0  units per week
Tables =  100.0  units per week


## Constraints
Again, don't forget `()` and look for any BINDING constraints where `LHS = RHS`.

Fabrication =  1850.0 hours
Assembly =  2075.0 hours
Shipping =  950.0 hours
Chairs =  0.0  units per week
Desks =  275.0  units per week
Tables =  100.0  units per week


<!-- oct8 -->
### LHS, RHS and slack, side by side

Printing the left-hand sides is a start, but it is much easier to spot a bottleneck when each constraint
sits next to its limit. The cell below makes a small table: what you **used** (LHS), the **limit** (RHS),
what is **left over** (slack = RHS - LHS), and a flag on the **binding** ones (slack = 0).

`value(c.upper)` reads the right-hand side of a `<=` constraint. For a `>=` constraint use `value(c.lower)`
instead, and the slack is LHS - RHS.

In [ ]:
# LHS, RHS and slack side by side - the BINDING ones are the bottlenecks
import pandas as pd

names = ["Fabrication hours", "Assembly hours", "Shipping hours", "Chair demand", "Desk demand", "Table demand"]
constraints = [model.constraint1, model.constraint2, model.constraint3, model.constraint4, model.constraint5, model.constraint6]

report = pd.DataFrame()
for i in range(len(constraints)):
    c = constraints[i]
    lhs = c()                  # left-hand side: what the plan actually uses
    rhs = value(c.upper)       # right-hand side: the limit (all six are <= constraints)
    slack = rhs - lhs          # what is left over
    if slack < 0.000001:
        status = "BINDING"
    else:
        status = ""
    row = pd.DataFrame({"constraint": [names[i]], "LHS (used)": [lhs], "RHS (limit)": [rhs], "slack": [slack], "binding?": [status]})
    report = pd.concat([report, row])

report.reset_index(drop=True, inplace=True)
report

<!-- oct8 -->
# Bonus: growing an expression with `+=` (a preview of soft-coding)

`x += something` means **the new value of `x` is the old value plus something new**. It is short for
`x = x + something`. Start from nothing and every pass through a loop tacks one more piece on. Run it and watch
it grow.

In [ ]:
# the += operator: new value = old value + something new
colors = ['red', 'green', 'blue', 'yellow']

expr = ''
for color in colors:
    expr += color + ' '      # same as: expr = expr + color + ' '
    print(expr)

In [ ]:
# the same trick builds this notebook's objective from a dictionary - nobody types 15*chairs + 24*desks + 18*tables
profit = {'chairs' : 15, 'desks' : 24, 'tables' : 18}

soft = ConcreteModel()
soft.x = Var(list(profit), domain=NonNegativeReals)

obj_expr = 0
for product in profit:
    obj_expr += profit[product] * soft.x[product]
    print(obj_expr)

<!-- oct8 -->
With 3 products the loop runs 3 times. With the 23 items on a McDonald's menu it runs 23 times, and you still
only typed it once. That is **soft-coding**, and Studio 3.1 shows the whole model built this way.

# All done!
Congrats! If you can repeat this (even just the bold headers), you are on your way to optimization modeling greatness. You may also choose to time your script or perform a sensitivty analysis on any binding constraints.

## Bottom line

- The Pyomo recipe never changes: **model → variables → objective → constraints → solve → read the results.**
- `maximize` OR `minimize`, never both. Don't forget the `()` when you read a value back (`model.profit()`).
- Check for **binding** constraints (where LHS = RHS) - those are the resources that are actually holding you back.

**Take it further:** swap the furniture data for a problem from your own world - a budget, a staffing plan, a menu - and reuse this exact skeleton. That's the whole point: one framework, endless problems.